# 02 - Python DSA Toolkit Refresher

Phase 0 of the DSA Roadmap. `list`/`dict`/`set` complexity is already covered in depth in `Python & DSA/02-data-structures-native.ipynb` -- this is a refresh, not a re-teach, plus the tools that notebook did not cover: `deque`, `heapq`, `bisect`, and the `itertools` used constantly for brute-force baselines.

## 30-second complexity drill (should be automatic before Phase 1)

| Structure | Index | Search | Insert/Delete at end | Insert/Delete at front |
|---|---|---|---|---|
| `list` | O(1) | O(n) | O(1) amortized | O(n) |
| `dict` / `set` | - | O(1) avg | O(1) avg | O(1) avg |
| `deque` | O(1) ends, O(n) middle | O(n) | O(1) | O(1) |

The one row that changes everything: `list` is fast at the end, slow at the front; `deque` is fast at BOTH ends. This single fact is why `deque` is the default choice for a queue (BFS -- Phase 2).

In [1]:
from collections import deque
import time

def list_front_ops(n):
    lst = []
    for i in range(n):
        lst.insert(0, i)      # O(n) each call -- shifts every existing element
    return lst

def deque_front_ops(n):
    dq = deque()
    for i in range(n):
        dq.appendleft(i)      # O(1) each call
    return dq

n = 8000
t0 = time.perf_counter(); list_front_ops(n); t_list = time.perf_counter() - t0
t0 = time.perf_counter(); deque_front_ops(n); t_deque = time.perf_counter() - t0
print(f"list.insert(0, x) x{n}:   {t_list:.4f}s")
print(f"deque.appendleft x{n}:    {t_deque:.4f}s")
print(f"speedup: {t_list/t_deque:.0f}x")

list.insert(0, x) x8000:   0.0077s
deque.appendleft x8000:    0.0003s
speedup: 28x


## heapq: a min-heap, built on a plain list

`heapq` operates on a regular Python list in place, keeping the smallest element at index 0. `heapify` is O(n); `heappush`/`heappop` are O(log n). Python only provides a MIN-heap directly -- for a max-heap, negate values going in and out.

In [2]:
import heapq

nums = [5, 1, 8, 3, 9, 2]
heapq.heapify(nums)                    # O(n), in place
print("heapified (index 0 is always the min):", nums)

heapq.heappush(nums, 0)
print("after push 0:", nums, "  smallest is still at index 0:", nums[0])
print("pop:", heapq.heappop(nums), " remaining:", nums)

# Classic pattern: k smallest / k largest without fully sorting
data = [7, 2, 9, 4, 1, 8, 3]
print("3 smallest:", heapq.nsmallest(3, data))
print("3 largest: ", heapq.nlargest(3, data))

# Max-heap via negation -- the standard workaround
max_heap = []
for x in [5, 1, 8, 3]:
    heapq.heappush(max_heap, -x)       # store negated
print("largest (max-heap via negation):", -max_heap[0])

heapified (index 0 is always the min): [1, 3, 2, 5, 9, 8]
after push 0: [0, 3, 1, 5, 9, 8, 2]   smallest is still at index 0: 0
pop: 0  remaining: [1, 3, 2, 5, 9, 8]
3 smallest: [1, 2, 3]
3 largest:  [9, 8, 7]
largest (max-heap via negation): 8


## bisect: binary search on a sorted list in one line

`bisect_left`/`bisect_right` find an insertion point in O(log n); `insort` inserts while keeping the list sorted (the insert itself is still O(n) for the shift, but finding WHERE is O(log n) instead of O(n)).

In [3]:
import bisect

sorted_nums = [1, 3, 3, 5, 7, 9]
print("bisect_left(3): ", bisect.bisect_left(sorted_nums, 3))    # leftmost valid insertion point for 3
print("bisect_right(3):", bisect.bisect_right(sorted_nums, 3))   # rightmost valid insertion point for 3
print("bisect_left(4): ", bisect.bisect_left(sorted_nums, 4))    # 4 isn't present -- still finds where it WOULD go

bisect.insort(sorted_nums, 4)
print("after insort(4):", sorted_nums, " -- stays sorted")

# Common pattern: "how many elements are <= x" in O(log n) instead of O(n) scan
print("count of elements <= 5:", bisect.bisect_right(sorted_nums, 5))

bisect_left(3):  1
bisect_right(3): 3
bisect_left(4):  3
after insort(4): [1, 3, 3, 4, 5, 7, 9]  -- stays sorted
count of elements <= 5: 5


## itertools: fast, correct brute-force baselines

Per the Problem-Solving Framework, step 4 is always "brute force first." These make the brute force force fast to write and hard to get subtly wrong -- and are sometimes the actual accepted answer for small constraints.

In [4]:
from itertools import combinations, permutations, product, accumulate

items = [1, 2, 3]
print("combinations(items, 2):", list(combinations(items, 2)))    # order doesn't matter, no repeats
print("permutations(items, 2):", list(permutations(items, 2)))    # order matters, no repeats
print("product(items, repeat=2):", list(product(items, repeat=2)))  # order matters, repeats allowed

nums = [1, 2, 3, 4]
print("accumulate (running sum):", list(accumulate(nums)))         # -- a prefix sum in one line
print("accumulate (running max):", list(accumulate(nums, func=max)))

combinations(items, 2): [(1, 2), (1, 3), (2, 3)]
permutations(items, 2): [(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2)]
product(items, repeat=2): [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3), (3, 1), (3, 2), (3, 3)]
accumulate (running sum): [1, 3, 6, 10]
accumulate (running max): [1, 2, 3, 4]


**Why this matters for Framework step 4 specifically:** `combinations`/`permutations` let a correct brute force be written in one line instead of hand-rolled recursive backtracking (Phase 3) -- useful to confirm the expected output on small examples before building the optimized version, and to have SOMETHING working if time runs out.

## Decision cheat sheet: which structure, for which DSA scenario

| Need | Reach for |
|---|---|
| "Have I seen this value before?" | `set` |
| Frequency counting | `dict` / `collections.Counter` |
| Fast add/remove at BOTH ends (queue, BFS) | `collections.deque` |
| Repeatedly need the min or max, with updates | `heapq` |
| Search / insertion point in a SORTED list | `bisect` |
| All subsets / orderings, as a brute-force baseline | `itertools` |
| Ordered key-value pairs, insertion order matters | plain `dict` (guaranteed ordered since 3.7) |
| Group values by a key without manual init-checks | `collections.defaultdict` |

## Practice

Implement each TODO using the tool named in the comment, then run the check cell.

In [5]:
def k_smallest(nums, k):
    # Return the k smallest values from nums, in ascending order. Use heapq.
    raise NotImplementedError

def sliding_window_max_positions(sorted_nums, target):
    # sorted_nums is sorted ascending. Return (lo, hi) = the bisect_left/bisect_right
    # positions for target, i.e. the [lo, hi) index range where target would sit.
    # Use bisect.
    raise NotImplementedError

def all_pairs_summing_to(nums, target):
    # Return all (a, b) pairs (as a set of frozensets, to ignore order/duplicates)
    # from nums that sum to target. Use itertools.combinations as the brute force.
    raise NotImplementedError

In [6]:
def _check(label, ok, detail=""):
    print(("[PASS] " if ok else "[FAIL] ") + label, detail)

try:
    r = k_smallest([7, 2, 9, 4, 1, 8, 3], 3)
    _check("k_smallest", r == [1, 2, 3], r)
except NotImplementedError:
    print("[SKIP] k_smallest -- not implemented yet")

try:
    r = sliding_window_max_positions([1, 3, 3, 5, 7, 9], 3)
    _check("sliding_window_max_positions", r == (1, 3), r)
except NotImplementedError:
    print("[SKIP] sliding_window_max_positions -- not implemented yet")

try:
    r = all_pairs_summing_to([1, 2, 3, 4, 5], 6)
    expected = {frozenset(p) for p in [(1, 5), (2, 4)]}
    _check("all_pairs_summing_to", r == expected, r)
except NotImplementedError:
    print("[SKIP] all_pairs_summing_to -- not implemented yet")

[SKIP] k_smallest -- not implemented yet
[SKIP] sliding_window_max_positions -- not implemented yet
[SKIP] all_pairs_summing_to -- not implemented yet


## Self-check before moving on

- [ ] I default to `deque` over `list` the instant front-of-structure operations are needed
- [ ] I can use `heapq` for k-largest/k-smallest without re-deriving the max-heap negation trick each time
- [ ] I can use `bisect` to turn an O(n) sorted-list scan into an O(log n) search
- [ ] I know at least 3 `itertools` functions well enough to write a one-line brute force
- [ ] Given a new problem, I can pick the right structure from the decision table above in under 10 seconds

Next: `03-recursion-fundamentals.ipynb`